# TMG-GAN Colab Workflow Notebook (Phase 1 + Phase 2)

This notebook only implements:
- Phase 1: Workflow contract (decision lock)
- Phase 2: Daily edit and push routine

It intentionally does not include training or evaluation cells yet.

In [ ]:
# Phase 1 - Workflow contract
import os
from pathlib import Path

CONTRACT = {
    "code_source_of_truth": "github",
    "runtime_branch": "main",
    "runtime_repo_mode": "reuse_and_pull",
    "dataset_policy": "runtime_if_complete_else_drive_zip",
    "allow_runtime_manual_code_edits": False,
}

REPO_URL = os.environ.get("TMG_REPO_URL", "https://github.com/Omayer111/TMG-GAN.git")
REPO_ROOT = Path(os.environ.get("TMG_REPO_ROOT", "/content/TMG-GAN"))
EXPECTED_BRANCH = CONTRACT["runtime_branch"]

print("Workflow contract locked:")
for key, value in CONTRACT.items():
    print(f"- {key}: {value}")
print("REPO_URL =", REPO_URL)
print("REPO_ROOT =", REPO_ROOT)

In [ ]:
# Shared helper for shell commands
import subprocess

def run(cmd: list[str], cwd: Path | None = None, check: bool = True) -> subprocess.CompletedProcess:
    print("+", " ".join(cmd))
    proc = subprocess.run(
        cmd,
        cwd=str(cwd) if cwd is not None else None,
        text=True,
        capture_output=True,
    )
    if proc.stdout:
        print(proc.stdout.rstrip())
    if proc.stderr:
        print(proc.stderr.rstrip())
    if check and proc.returncode != 0:
        raise RuntimeError(f"Command failed ({proc.returncode}): {' '.join(cmd)}")
    return proc

In [ ]:
# Phase 1 - Runtime contract checks
if not REPO_ROOT.exists():
    raise FileNotFoundError(
        f"Runtime repo not found: {REPO_ROOT}. Clone it first before continuing."
    )

origin_url = run(["git", "-C", str(REPO_ROOT), "remote", "get-url", "origin"], check=False).stdout.strip()
current_branch = run(["git", "-C", str(REPO_ROOT), "rev-parse", "--abbrev-ref", "HEAD"], check=False).stdout.strip()

print("Current origin URL:", origin_url)
print("Current branch:", current_branch)

if origin_url and origin_url != REPO_URL:
    print("WARNING: origin URL differs from expected REPO_URL.")

if current_branch != EXPECTED_BRANCH:
    print(f"WARNING: expected branch '{EXPECTED_BRANCH}' but found '{current_branch}'.")
else:
    print("Branch contract satisfied.")

## Phase 2 - Daily Edit and Push Routine

Run this sequence every training day:
1. Edit code in VS Code.
2. `git add -A`
3. `git commit -m "..."`
4. `git push origin main`
5. Capture pushed commit hash and set `TMG_EXPECTED_COMMIT` before runtime sync phases.

Rule: do not start training if your intended commit is not pushed.

In [ ]:
# Phase 2 - Pre-push safety check
status = run(["git", "-C", str(REPO_ROOT), "status", "--porcelain"], check=False).stdout.strip()
last_commit = run(["git", "-C", str(REPO_ROOT), "log", "-1", "--oneline"], check=False).stdout.strip()

print("Last commit:", last_commit or "<none>")

if status:
    lines = status.splitlines()
    preview = "\n".join(lines[:20])
    print("Working tree is NOT clean. Pending changes:")
    print(preview)
    print("Next action: commit and push before runtime execution phases.")
else:
    print("Working tree is clean. You can proceed to commit hash capture.")

In [ ]:
# Phase 2 - Commit lock for later runtime sync
EXPECTED_PUSHED_COMMIT = os.environ.get("TMG_EXPECTED_COMMIT", "").strip()
runtime_head = run(["git", "-C", str(REPO_ROOT), "rev-parse", "--short", "HEAD"], check=False).stdout.strip()

print("Runtime HEAD:", runtime_head or "<unknown>")
if not EXPECTED_PUSHED_COMMIT:
    print("Set TMG_EXPECTED_COMMIT to your pushed short hash before Phase 3 notebook steps.")
else:
    print("TMG_EXPECTED_COMMIT =", EXPECTED_PUSHED_COMMIT)
    if runtime_head == EXPECTED_PUSHED_COMMIT:
        print("Commit lock check passed.")
    else:
        print("Commit lock check not yet passed (this is expected before runtime pull phase).")

## Stop Here

Phase 1 and Phase 2 are complete in this notebook.
Next notebook step (when you ask): Phase 3 runtime sync and integrity gates.